# U2-04 · Árboles, SVM, redes neuronales y conjuntos de modelos

*Machine Learning aplicado usando Python (IFCD093PO) · Unidad 2, apartados 2.4 y 2.5*

Ejecuta cada celda con **Mayús + Intro**. En el navegador los modelos tardan más que en un ordenador con Python instalado: la SVM y el bosque aleatorio, alrededor de un minuto cada uno.

> Datos (UCI, licencia CC BY 4.0), junto a este cuaderno; su origen se explica en `LEEME.md`:
> Janosi, A., Steinbrunn, W., Pfisterer, M. y Detrano, R. (1989). *Heart Disease*. https://doi.org/10.24432/C52P4X ·
> Hofmann, H. (1994). *Statlog (German Credit Data)*. https://doi.org/10.24432/C5NC77 ·
> Sakar, C. O. y Kastro, Y. (2018). *Online Shoppers Purchasing Intention Dataset*. https://doi.org/10.24432/C5F88Q ·
> Chen, D. (2012). *Online Retail II*. https://doi.org/10.24432/C5CG6D

## 1. Caso de empresa: ¿comprará esta visita? (2.4)

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

d = pd.read_csv("compras.csv")                 # 12 330 visitas
y = d.pop("Revenue").astype(int)               # 1 = acabó en compra
cat = ["Month", "VisitorType", "Weekend", "OperatingSystems",
       "Browser", "Region", "TrafficType"]
num = d.columns.difference(cat)

def prep(escalar):              # los árboles no necesitan escalar
    return make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         cat),
        (StandardScaler() if escalar else "passthrough", num),
        verbose_feature_names_out=False)

X_ent, X_pru, y_ent, y_pru = train_test_split(
    d, y, test_size=0.3, stratify=y, random_state=0)

def evalua(nombre, mod):
    """Entrena y muestra el acierto equilibrado y el AUC en prueba."""
    t = time.perf_counter()
    mod.fit(X_ent, y_ent)
    if hasattr(mod, "decision_function"):            # la SVM
        s = mod.decision_function(X_pru)
    else:                                            # los demás
        s = mod.predict_proba(X_pru)[:, 1]
    eq = balanced_accuracy_score(y_pru, mod.predict(X_pru))
    print(f"{nombre:<22} entrenamiento {mod.score(X_ent, y_ent):.3f}"
          f" ({time.perf_counter() - t:.0f} s)")
    print(f"   prueba: acierto equilibrado {eq:.3f}, "
          f"AUC {roc_auc_score(y_pru, s):.3f}")
    return mod

if __name__ == "__main__":
    print(f"compran {y.mean():.1%} de las visitas")
    arbol = evalua("árbol, profundidad 3", make_pipeline(
        prep(False), DecisionTreeClassifier(max_depth=3,
                                            random_state=0)))
    evalua("árbol, sin límite", make_pipeline(
        prep(False), DecisionTreeClassifier(random_state=0)))
    evalua("SVM (núcleo RBF)", make_pipeline(prep(True), SVC()))
    evalua("red neuronal 32-16", make_pipeline(
        prep(True), MLPClassifier((32, 16), early_stopping=True,
                                  max_iter=500, random_state=0)))
    plt.figure(figsize=(5.5, 2.4))
    plot_tree(arbol[-1], max_depth=2, fontsize=5,
              feature_names=arbol[0].get_feature_names_out(),
              class_names=["no compra", "compra"], filled=True,
              impurity=False, proportion=True)
    plt.show()  # plt.savefig("arbol.pdf", bbox_inches="tight")

**Prueba.** Entrena árboles de profundidad 2, 4, 6 y 8. ¿En qué profundidad empieza a empeorar la prueba?

## 2. Bosque aleatorio, gradient boosting y votación (2.5)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

bosque = evalua("bosque aleatorio (500)", make_pipeline(
    prep(False), RandomForestClassifier(500, min_samples_leaf=3,
                                        n_jobs=-1, random_state=0)))
evalua("gradient boosting", make_pipeline(
    prep(False), HistGradientBoostingClassifier(random_state=0)))
votos = VotingClassifier([
    ("bosque", RandomForestClassifier(500, min_samples_leaf=3,
                                      n_jobs=-1, random_state=0)),
    ("logistica", LogisticRegression(max_iter=1000))], voting="soft")
evalua("votación (2 modelos)", make_pipeline(prep(True), votos))

# ¿qué variables usa el bosque? Se barajan una a una en prueba
imp = permutation_importance(bosque, X_pru, y_pru, scoring="roc_auc",
                             n_repeats=10, random_state=0)
imp = pd.Series(imp.importances_mean, index=X_pru.columns)
imp = imp.sort_values().tail(6)
print((imp * 100).round(1).to_string())       # puntos de AUC perdidos
imp.plot.barh(figsize=(4.4, 1.8))
plt.xlabel("pérdida de AUC al barajar la variable")
plt.show()  # plt.savefig("importancia.pdf")

## 3. Predicción avanzada de la serie de ventas (2.5)

La primera celda repite la preparación del apartado 2.3.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, r2_score,
                             root_mean_squared_error)

v = pd.read_csv("ventas_diarias.csv", parse_dates=["fecha"],
                index_col="fecha")["ventas"] / 1000  # miles de £

def variables(v):
    """Lo que se sabe la víspera: ventas pasadas y el calendario."""
    X = pd.DataFrame({"ayer": v.shift(1), "hace_7": v.shift(7),
                      "media_7": v.shift(1).rolling(7).mean(),
                      "media_28": v.shift(1).rolling(28).mean()})
    dias = pd.get_dummies(v.index.dayofweek, prefix="dia",
                          dtype=float).set_index(v.index)
    return X.join(dias).dropna()

X = variables(v)
y = v.loc[X.index]
ent, pru = X.index < "2011-07-01", X.index >= "2011-07-01"
print(f"entrenamiento {ent.sum()} días, prueba {pru.sum()} días")

def mide(nombre, pred):
    yp = y[pru]
    print(f"{nombre:<26} EAM {mean_absolute_error(yp, pred):5.2f}  "
          f"RECM {root_mean_squared_error(yp, pred):5.2f}  "
          f"R2 {r2_score(yp, pred):5.2f}")

if __name__ == "__main__":
    mide("referencia: hace 7 días", X.loc[pru, "hace_7"])
    lineal = LinearRegression().fit(X[ent], y[ent])
    mide("regresión lineal", lineal.predict(X[pru]))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (RandomForestRegressor,
                              HistGradientBoostingRegressor)

X = X.assign(mes=X.index.month)                  # estación del año
modelos = {"regresión lineal": LinearRegression(),
           "bosque aleatorio": RandomForestRegressor(
               500, min_samples_leaf=3, random_state=0, n_jobs=-1),
           "gradient boosting": HistGradientBoostingRegressor(
               max_iter=300, learning_rate=0.05, random_state=0)}
mide("referencia: hace 7 días", X.loc[pru, "hace_7"])
pred = {}
for nombre, m in modelos.items():
    pred[nombre] = m.fit(X[ent], y[ent]).predict(X[pru])
    mide(nombre, pred[nombre])
media = (pred["bosque aleatorio"] + pred["gradient boosting"]) / 2
mide("media de los dos conjuntos", media)

fig, eje = plt.subplots(figsize=(4.4, 2.0))
eje.plot(y[pru].index, y[pru], lw=0.8, label="ventas reales")
eje.plot(y[pru].index, media, lw=0.8, label="previsión")
eje.set_ylabel("miles de £ al día"); eje.legend()
fig.autofmt_xdate(); plt.show()  # fig.savefig("ventas.pdf")

**Ejercicio.** Cambia el umbral de decisión del bosque aleatorio de la sección 2 para que su acierto equilibrado sea máximo. ¿Supera al árbol de profundidad 3?

> Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324